# Moon Rover MPC Testing

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import functools
import jax.numpy as jnp
import numpy as np
import tqdm
import pandas as pd

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.quartic_cost as quartic_cost

In [ ]:
def load_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    """Load reference linear accelerations and angular velocities."""
    df = pd.read_csv(file_path)

    acc_keys = [f"sesmt.md.merged_frame.xyz_acc[{i}] {{m/s^2}}" for i in range(3)]
    omega_keys = [f"sesmt.md.merged_frame.ang_vel[{i}] {{rad/s}}" for i in range(3)]
    gravity_keys = [
        f"sesmt.md.merged_frame.gravity[{i}] {{m/s^2}}" for i in range(3)
    ]

    acc_ref = jnp.array(df[acc_keys])
    omega_ref = jnp.array(df[omega_keys])
    gravity_ref = jnp.array(df[gravity_keys])

    # for some reason, data collection after a lot of nonsense data
    # we grab the data after we start recognizing nonzero (x) accelerations
    # note that using direct equality is desired here (and not jnp.isclose)
    offset = jnp.argmax(acc_ref[:, 0] != 0.0)

    # we need to cancel and then add back in the gravity vector
    acc_ref = acc_ref[offset:, :] - 2 * gravity_ref[offset:, :]
    omega_ref = omega_ref[offset:, :]
    
    return acc_ref, omega_ref


def load_specific_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    df = pd.read_csv(file_path)

    ks = df.keys()

    ts = np.array(df[ks[0]])
    diff = np.abs(np.diff(ts))
    avg_diff = np.mean(diff)
    std_diff = np.std(diff)
    bad_indices = np.where(diff > avg_diff + std_diff)[0] + 1 # off by one
    start_index = bad_indices[-2] + 5 * 200
    end_index = bad_indices[-1] - 1

    # ts = ts[start_index: end_index + 1]
    df = df[start_index: end_index + 1]

    acc_ref = jnp.transpose(jnp.array([df[k] for k in ks[1:4]]))
    omega_ref = jnp.transpose(jnp.array([df[k] for k in ks[4:]]))
    return acc_ref, omega_ref

In [ ]:
# load data

# file_path = "/Users/jozbee/work/eng/comp/data/00_sms_drive.csv"
# acc_ref, omega_ref = load_sms_references(file_path)

file_path = "/Users/jozbee/work/eng/comp/data/specific-forces-standard-road-v2.csv"
acc_ref, omega_ref = load_specific_sms_references(file_path)

assert acc_ref.shape[0] == omega_ref.shape[0]
assert acc_ref.shape[1] == 3
assert omega_ref.shape[1] == 3

In [ ]:
# general setup
begin = 0
num_steps = 100 * 200
n = 200  # horizon

In [ ]:
# cost setup

# weights = opt.ExpWeights(
#     acc=jnp.array([5.0, 5.0, 1.0]),
#     omega=jnp.array([1.0, 1.0, 5.0]),
#     alpha_acc=jnp.array(1.0),
#     alpha_omega=jnp.array(4.0),
# )

weights = opt.ExpWeights(
    acc=jnp.array([1e1, 1e1, 1e0]),
    omega=jnp.array([1e1, 1e1, 1e1]),
    alpha_acc=jnp.array([0.0]),
    alpha_omega=jnp.array([0.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]
euler_margins = [0.2 / 3.0, 0.1 / 3.0]
euler_sizes = [2**0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    # margins=[0.6],
    # sizes=[1.0, 25],
    # margins=euler_margins,
    # sizes=euler_sizes,
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes = sizes,
    # margins=euler_margins,
    # sizes=euler_sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    # margins=euler_margins,
    # sizes=euler_sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
roll_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_roll,
    high=const.max_roll,
)
pitch_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_pitch,
    high=const.max_pitch,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_rotary_yaw,
    high=const.max_rotary_yaw,
)
yaw_dot_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_rotary_vel,
    high=const.max_rotary_vel,
)
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    roll_cost=roll_cost,
    pitch_cost=pitch_cost,
    yaw_cost=yaw_cost,
    yaw_dot_cost=yaw_dot_cost,
)

In [ ]:
# run setup
train_step = functools.partial(opt.train_step_with_cost, weights, cost_terms)

train_state = opt.TrainState.zero_init(n, ("earth", "moon"))
train_list = []
times = []
sol_list = []
res_list = []

In [ ]:
# run
for i in tqdm.tqdm(range(num_steps)):
    aref = jnp.tile(acc_ref[begin + i], reps=(n, 1))
    oref = jnp.tile(omega_ref[begin + i], reps=(n, 1))
    train_state, sol, res, t_tot = train_step(
        aref, oref, train_state, opt_options={"maxiter": 4, "maxls": 1}
    )
    train_list.append(train_state)
    sol_list.append(sol)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
bad = 51 * 200 + 95

In [ ]:
# y_states = jnp.array([sol.vstate_irl.y_state[0] for sol in sol_list[:bad]])
# y_states = jnp.array([sol.vstate_sim.y_state[0] for sol in sol_list[:bad]])
y_states = sol_list[bad].vstate_sim.y_state

In [ ]:
sol_list[bad].vstate_sim.x_state[-1]

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
# ax.plot(acc_ref[:, 0])
# ax.plot(acc_ref[:, 1])
# ax.plot(acc_ref[:, 3])
ax.plot(y_states[:, 0])
ax.plot(y_states[:, 1])
# ax.plot(y_states[:, 2])
ax.plot(y_states[:, 3])
ax.plot(y_states[:, 4])
ax.plot(y_states[:, 5])
ax.grid()
plt.show()

In [ ]:
# plot_size = 51 * 200 + 95
plot_size = num_steps

trajectory = sol_list[:plot_size]
references = {
    "xyz-acceleration": jnp.array(acc_ref[begin: begin + plot_size]),
    "angular-velocity": jnp.array(omega_ref[begin: begin + plot_size]),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=trajectory, references=references)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=trajectory)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=trajectory)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=trajectory)

In [ ]:
assert False

In [ ]:
# plot cost history
get_cost_ts = functools.partial(
    opt.get_scipy_cost_ts,
    weights,
    cost_terms,
    # acc_ref,
    # omega_ref,
    use_rotary=True,
)

costs = []
for state in train_list:
    cost_fun, _ = get_cost_ts(
        acc_ref=jnp.tile(acc_ref[begin + i], (n, 1)),
        omega_ref=jnp.tile(omega_ref[begin + i], (n, 1)),
        train_state=state,
    )
    costs.append(cost_fun(np.array(state.control_flat)))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.set_title("Cost history")
ax.set_xlabel("iter")
ax.set_ylabel("cost")
ax.plot(costs)
ax.grid()
fig.tight_layout()
plt.show()

In [ ]:
# assert False

## animations

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
# mp_mpl.call_mp_animate_trajectory(
#     file_name="data/sms_acc_3d.mp4",
#     trajectory=trajectory,
# )

In [ ]:
# mp_mpl.call_mp_animate_human_trajectory(
#     file_name="data/sms_acc_human.mp4",
#     trajectory=trajectory,
#     references=references,
# )

In [ ]:
# def cost_trajectory_tile(arr: jax.Array) -> jax.Array:
#     arr = arr[begin: begin + num_steps]
#     arr = arr.reshape(arr.shape[0], 1, arr.shape[1])
#     return jnp.tile(arr, (1, n, 1))

# mp_mpl.call_mp_animate_cost_trajectory(
#     file_name="data/sms_acc_cost.mp4",
#     acc_refs=cost_trajectory_tile(acc_ref),
#     omega_refs=cost_trajectory_tile(omega_ref),
#     weights=weights,
#     cost_terms=cost_terms,
#     trajectory=trajectory,
# )